# 14 — Mars volcanic and basaltic terrain process gate

This notebook is process-specific. It does not convert a cluster into a process label. It checks whether the Mars inputs contain the evidence required by the reviewed process specification. If evidence is missing, it writes an inconclusive result and stops before Earth matching.


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROCESS_ID = 'volcanic_basaltic_terrain'
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
CONFIG_CANDIDATES = list(Path('/kaggle/input').rglob('mars_process_specifications.json')) if Path('/kaggle/input').exists() else []
CONFIG = CONFIG_CANDIDATES[0] if CONFIG_CANDIDATES else Path('configs/mars_process_specifications.json')
if not CONFIG.exists():
    raise FileNotFoundError('Attach the repository/configuration input containing mars_process_specifications.json')
spec = next(x for x in json.loads(CONFIG.read_text()) if x['process_id'] == PROCESS_ID)
files = list(ROOT.rglob('mars_candidate_validation_results.parquet'))
if not files:
    raise FileNotFoundError('Attach the completed Mars candidate-validation output before running this notebook')
candidates = pd.read_parquet(files[0])
required = spec.get('mars_evidence_variables', [])
available = [x for x in required if x in candidates.columns]
status = spec.get('status', 'unknown')
missing = [x for x in required if x not in candidates.columns]
if status != 'supported_for_earth_screening' or missing:
    decision = 'inconclusive_pending_mars_evidence'
    reason = 'The process specification is not Earth-screening-ready or required Mars columns are absent.'
else:
    decision = 'ready_for_specialist_review'
    reason = 'Required named columns exist; scientific review and independent validation are still required.'
result = {'process_id': PROCESS_ID, 'decision': decision, 'reason': reason, 'required_variables': required, 'available_named_columns': available, 'missing_named_columns': missing, 'candidate_rows': int(len(candidates))}
OUT = ROOT / f'{PROCESS_ID}_process_output'; OUT.mkdir(exist_ok=True)
(OUT / 'process_readiness.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
pd.DataFrame([result]).to_csv(OUT / 'process_readiness.csv', index=False)
print(json.dumps(result, indent=2))
if decision != 'ready_for_specialist_review':
    print('No Earth matching was performed: this process is inconclusive.')
